In [3]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import warnings
import sys

In [6]:
# Supress Tensorflow warnings
warnings.filterwarnings('ignore', category=FutureWarning)
tf.get_logger().setLevel('ERROR')


# ---- Load Prep Data

path = "F:/Users/MSR/Documents/CST4200/15/Datasets/student-mat.csv"
df0 = pd.read_csv(path, sep=";")

# --- Read CSV File
print(f'Successfully loaded file: {path}\n')

# --- Display Column Type
print(" -- Column Info Initial -- ")
df0.info()

Successfully loaded file: F:/Users/MSR/Documents/CST4200/15/Datasets/student-mat.csv

 -- Column Info Initial -- 
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 395 entries, 0 to 394
Data columns (total 33 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   school      395 non-null    object
 1   sex         395 non-null    object
 2   age         395 non-null    int64 
 3   address     395 non-null    object
 4   famsize     395 non-null    object
 5   Pstatus     395 non-null    object
 6   Medu        395 non-null    int64 
 7   Fedu        395 non-null    int64 
 8   Mjob        395 non-null    object
 9   Fjob        395 non-null    object
 10  reason      395 non-null    object
 11  guardian    395 non-null    object
 12  traveltime  395 non-null    int64 
 13  studytime   395 non-null    int64 
 14  failures    395 non-null    int64 
 15  schoolsup   395 non-null    object
 16  famsup      395 non-null    object
 17  paid        395 

In [12]:
# --- Introduce new column 'High_G3' ---
# G3 is the final grade
df = df0.copy()
df['High_G3']  = np.where(df['G3'] > 10, 'High', 'Moderate')

# --- Print Distribution of 'High_G3' ---
print('--- Distribution of New Target Column "High_G3" ---')
print(df['High_G3'].value_counts())

# --- Remove G3 from Column
df = df.drop('G3', axis=1)

# --- Prepare x(features) and y(target)
y = df['High_G3']
x = df.drop('High_G3', axis=1)

# --- Convert CAtegorical Features into one hot Encoding
#  'yes'/'No' columns
#  Find Object type columns to encode
categorical_cols = x.select_dtypes(include=['object']).columns

# Use pd.get_dummies for one hot encoding
x_processed = pd.get_dummies(x, columns=categorical_cols, drop_first=True)

print("---- Preprocessing --")
print(f'Original Featurer Count: {len(x.columns)}')
print(f'Feature Count after one-hot-encoddig: {len(x_processed.columns)}')

--- Distribution of New Target Column "High_G3" ---
High_G3
High        209
Moderate    186
Name: count, dtype: int64
---- Preprocessing --
Original Featurer Count: 32
Feature Count after one-hot-encoddig: 41


In [13]:
# Standard Scaler Features
scaler = StandardScaler()
x_scl = scaler.fit_transform(x_processed)

# --- Perform test-train split 80-90
y_numerical = y.map({'Moderate':0, 'High':1})

xTrn, xTst, yTrn, yTst = train_test_split(x_scl, y_numerical, test_size=0.2, random_state=63, stratify=y_numerical) #(helps keep distribution in test/train)

print(f"Training Sample: {xTrn.shape[0]}")
print(f"Testing Sampels: {xTst.shape[0]}")

Training Sample: 316
Testing Sampels: 79


In [16]:
#  --- Fit ML: Perceptron using Sequential

# Get number of input features after processng
nFeat = xTrn.shape[1]

#  Initialize the Model
model = Sequential()

# Add input layer and first hidden layer (32-neurons)
model.add(Dense(32, activation='relu', input_shape=(nFeat,)))

# Adds second hidden layer (16 neurons)
model.add(Dense(16, activation='relu'))

# Add output layer, (1 neuron, sigmoid for binary classification)
model.add(Dense(1, activation='sigmoid'))

# Compile the model
model.compile(loss='mean_squared_error', metrics=['accuracy'])

print("--- Model Summary ---")
model.summary()

--- Model Summary ---
Model: "sequential_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_6 (Dense)             (None, 32)                1344      
                                                                 
 dense_7 (Dense)             (None, 16)                528       
                                                                 
 dense_8 (Dense)             (None, 1)                 17        
                                                                 
Total params: 1,889
Trainable params: 1,889
Non-trainable params: 0
_________________________________________________________________


In [18]:
# Train Fit the Model
print("--- Training Model ---")
history = model.fit(
    xTrn,
    yTrn,
    epochs=50,                    # Number of passes through the data 
    batch_size=16,                # Number of samples per update
    validation_data=(xTst, yTst),
    verbose=1    # Show progress
)

print("Model Training Complete")

--- Training Model ---
Epoch 1/50
20/20 [==============================] - 2s 18ms/step - loss: 0.2667 - accuracy: 0.5411 - val_loss: 0.3212 - val_accuracy: 0.4810
Epoch 2/50
20/20 [==============================] - 0s 11ms/step - loss: 0.2212 - accuracy: 0.6519 - val_loss: 0.2961 - val_accuracy: 0.5316
Epoch 3/50
20/20 [==============================] - 0s 10ms/step - loss: 0.1935 - accuracy: 0.6994 - val_loss: 0.2756 - val_accuracy: 0.5949
Epoch 4/50
20/20 [==============================] - 0s 11ms/step - loss: 0.1705 - accuracy: 0.7785 - val_loss: 0.2558 - val_accuracy: 0.6456
Epoch 5/50
20/20 [==============================] - 0s 11ms/step - loss: 0.1495 - accuracy: 0.8133 - val_loss: 0.2365 - val_accuracy: 0.6709
Epoch 6/50
20/20 [==============================] - 0s 10ms/step - loss: 0.1303 - accuracy: 0.8386 - val_loss: 0.2195 - val_accuracy: 0.6709
Epoch 7/50
20/20 [==============================] - 0s 10ms/step - loss: 0.1141 - accuracy: 0.8608 - val_loss: 0.2025 - val_accurac

In [ ]:
# Step-by-Step Explanation

## 1. Original Data ("Column Info Initial")

You started with a **DataFrame of 395 rows × 33 columns** — the classic UCI Student Performance dataset.

- **16 integer columns**: numeric features like `age`, `Medu`, `studytime`, `absences`, and the grades `G1`, `G2`, `G3`.
- **17 object (string/categorical) columns**: things like `school`, `sex`, `address`, `Mjob`, `Fjob`, `internet`, etc.

Key point: every column has **395 non-null values** → there are **no missing values** to impute. That's why preprocessing could focus on encoding rather than cleaning.

The target `G3` (final grade, 0–20) is still numeric at this stage.

## 2. New Target Column "High_G3" (Binarization / Class Creation)

Instead of predicting the raw `G3` score, you converted it into a **two-class label**:

| Class | Count |
|-------|-------|
| High  | 209 |
| Moderate | 186 |

- `G3` was **discretized** into two categories (likely a threshold, e.g., `G3 >= 10` → "High", else "Moderate").
- Total = 209 + 186 = **395** ✓ matches the original row count.
- The classes are **fairly balanced** (~53% vs ~47%), which is good — no need for heavy resampling like SMOTE.

This turns the problem from **regression** into **binary classification**.

## 3. Preprocessing ("Original Feature Count: 32 → 41")

- **Original feature count = 32**: the 33 columns minus the target column (`G3`, which was replaced by `High_G3`). So 32 input features.
- **After one-hot encoding = 41**: categorical columns were expanded into dummy/indicator columns via **one-hot encoding** (e.g., `sex` → `sex_M`, `sex_F`; `Mjob` → `Mjob_at_home`, `Mjob_health`, …).

Why 41 and not more? Because scikit-learn/pandas one-hot encoding typically uses `drop_first=True`, dropping one dummy per categorical variable to avoid the **dummy variable trap** (perfect multicollinearity). So the net increase is only 41 − 32 = **9 extra columns** despite 17 categorical columns.

Result: a numeric feature matrix of shape **(395, 41)** ready for a neural network.

## 4. Model Summary ("Model: sequential_2")

This is a **Keras Sequential model** — a plain feed-forward neural network (Multilayer Perceptron). Note the name `sequential_2` suggests this is the second model instance built in the session.

Let's walk through each layer:

### Layer 1: `dense_6` — Input → Hidden
- **Output shape: (None, 32)** → 32 neurons.
- **Param #: 1344**
- Math: `(41 input features × 32 neurons) + 32 biases = 1312 + 32 = 1344` ✓
- So the input dimension is confirmed as **41** (matching the one-hot encoded features).

### Layer 2: `dense_7` — Hidden → Hidden
- **Output shape: (None, 16)** → 16 neurons.
- **Param #: 528**
- Math: `(32 × 16) + 16 = 512 + 16 = 528` ✓

### Layer 3: `dense_8` — Output Layer
- **Output shape: (None, 1)** → **1 neuron**.
- **Param #: 17**
- Math: `(16 × 1) + 1 = 16 + 1 = 17` ✓
- A **single output neuron** with (almost certainly) a **sigmoid activation** → outputs a probability of belonging to the positive class ("High"). This confirms **binary classification** with binary cross-entropy loss.

### Totals
- **Total params: 1,889** (= 1344 + 528 + 17)
- **Trainable: 1,889 / Non-trainable: 0** → no frozen layers; the whole network learns.

## 5. Architecture Summary

```
Input (41 features)
   ↓ Dense(32) + activation (likely ReLU)
   ↓ Dense(16) + activation (likely ReLU)
   ↓ Dense(1)  + sigmoid
Output: P(High_G3)
```

This is a small, lightweight MLP (~1.9K parameters) — appropriate for a dataset of only 395 samples, where huge networks would overfit.

## 6. Note on "Perceptron"

Although you called it a Perceptron, technically a **single perceptron** is one neuron with a step function. What you built is a **multilayer perceptron (MLP)** — a neural network with hidden layers. The naming in Keras (`dense` layers) reflects this. It's the modern, gradient-descent-trained generalization of the perceptron.

## Summary Flow

| Stage | What happened | Result |
|-------|---------------|--------|
| Raw data | 395 × 33, no NaNs | Clean dataset |
| Target engineering | `G3` → `High_G3` (binary) | 209 High / 186 Moderate |
| Preprocessing | One-hot encode categoricals | 32 → 41 features |
| Model | MLP: 41 → 32 → 16 → 1 | 1,889 trainable params |
| Task | Binary classification | Predict P(High_G3) |

In [ ]:
# Real-World Interpretation of the Training Log

## 1. What the numbers say at a glance

| ---- Metric ----- | --- Training - |  ------- Validation ----------- |
|-------------------|----------------|---------------------------------|
| Final accuracy    | **100%**       | **~86%**                        |
| Final loss        | **~0.0000035** | **~0.119**                      |
| Best val accuracy | —              | **86.08%** (epochs 45–50)       |
| Best val loss     | —              | **~0.115** (around epoch 20–24) |

The model learned the **training data perfectly** (it memorized it), 
                    but on **unseen data** it tops out around **86% accuracy**. 
                        That 14-point gap between train and validation is the classic signature of **overfitting**.

## 2. What "86% validation accuracy" means in the real world

Imagine deploying this model to a school to flag students likely to finish with a **"High"** final grade (vs. "Moderate").

- If you fed it **100 new students it has never seen**, it would sort roughly **86 correctly and 14 incorrectly**.
- With the class split ~53% High / 47% Moderate, a naive "always predict High" baseline gets only ~53%. 
                                                So **86% is genuinely useful** — it's ~33 points better than guessing.

But 86% is an **average**. Because the classes are near-balanced, a single accuracy number hides *which* mistakes it makes. In practice you'd want to know:
- **Precision/recall per class** — does it miss "Moderate" students (false "High" predictions) more than it misses "High" ones?
- A student wrongly predicted "High" might be **denied an intervention** they needed. 
    A student wrongly predicted "Moderate" might get **unnecessary support**. 
    The cost of each error type matters more than raw accuracy.

## 3. The overfitting story (the most important real-world lesson)

Look at the trajectory:

- **Epochs 1–12**: train and val accuracy both climb together (54% → 94% train, 48% → 82% val). Healthy learning.
- **Epochs 13–22**: train accuracy shoots to ~99%, val plateaus at ~85%. **The model starts memorizing.**
- **Epochs 23–50**: train loss collapses to near-zero (3.5e-6), train accuracy hits 100%, but **val loss stops improving and even drifts up slightly** (0.115 → 0.119). 
                    This is textbook overfitting — the model is fitting noise in the 395 training samples.

**Real-world consequence:** The 100% training accuracy is **meaningless** — it's a memorization artifact. 
                            The honest number to quote is **~86% on unseen students**. 
                                Anyone reporting "100% accuracy" to stakeholders would be misleading them.

Also note **val loss is lowest around epoch 20–24 (~0.115)**, while val accuracy peaks later at 86%. 
                This means the best *generalizing* model was likely reached **around epoch 20**, and training to 50 made it slightly worse-calibrated (more confident on wrong answers). 
                    **Early stopping** would have been the right move.

## 4. What the model is actually doing

A small MLP (41 → 32 → 16 → 1) with ~1,889 parameters on only **395 samples** — that's roughly **5 parameters per sample**, which is a lot for a dataset this small. 
The network has enough capacity to fit every training point, which is exactly what happened.

Real-world translation: **the model has learned the training students, not the underlying pattern of student success.
                        ** It will do fine on students similar to the training set, but its confidence on borderline cases is unreliable.

## 5. Practical real-world caveats you must state

1. **Small, single-school dataset** — 395 students from one Portuguese school. 
                                        The 86% will **not** transfer directly to a different school, country, or demographic. 
                                        Expect degradation on deployment.
2. **The target is coarse** — "High vs Moderate" is a binary cut of a 0–20 grade. It doesn't tell you *how* high, and the threshold choice (which defines "High") heavily affects results.
3. **No fairness check shown** — with features like `sex`, `address`, `Mjob`, `Fjob`, `internet`, the model could encode **socioeconomic or demographic bias**. 
                                Before real use, you'd need to check error rates across groups.
4. **Correlation ≠ causation** — features like `goout`, `Dalc`, `Walc`, `absences` are associated with grades, but the model can't tell you that changing them *causes* a grade change. 
                                 It's a **risk flag**, not a prescription.
5. **The 86% is on a random split of 395 rows** — the validation set is only ~80 students (20/20 batches ⇒ ~20 batches of ~20 ⇒ ~395 total, val ≈ 20%). 
                                                  **86% ± a few percent** is within noise; a different split could give 82% or 89%.

## 6. How to use it responsibly in the real world

- **Report 86%, not 100%.** Quote validation accuracy and loss, and mention the overfitting gap.
- **Use it as a triage/early-warning tool**, not a decision-maker. Flag at-risk students for a human counselor to review.
- **Add early stopping / dropout / L2 regularization** and retrain — you'd likely keep ~86% val accuracy while closing the train/val gap, making the model more trustworthy.
- **Validate on a held-out school or a temporally later cohort** before claiming real-world performance.
- **Check per-class recall** so you know whether it's better at catching "High" or "Moderate" students — that determines whether it's safe to use for intervention targeting.

## Bottom line

> The model **learned its training set perfectly (100%) but generalizes to about 86% on unseen students** — a modest, honest, and genuinely useful result for a tiny 395-row dataset, 
                                                                                                            undermined by clear overfitting after ~epoch 20. 
                                                                             In the real world it's a **decent early-warning screener**, 
                                                                             not a reliable predictor, and any "100% accuracy" claim would be an artifact of memorization rather than real predictive skill.